In [1]:
import importlib
import cleaning
importlib.reload(cleaning)
from cleaning import clean_df, print_verification
import pandas as pd

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# impor modules and read the data


# Read the data from the CSV file (using absolute path)
df = pd.read_csv('/home/sheikh/Projects/Thesis/data/medium.csv')

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:\n", df.head(3))

Shape: (1165, 3)

Dtypes:
 grmd         int64
md_name     object
tex_text    object
dtype: object

First 3 rows:
    grmd                    md_name  \
0  1436   LIMNOCHORDIA L945 MEDIUM   
1  1479   SOLIDESULFOVIBRIO MEDIUM   
2  1481  Bold's Basal Medium (BBM)   

                                            tex_text  
0  \mono{KH$_2$PO$_4$} {0.2} {g}\n\mono{MgCl$_2$$...  
1  \mono{KH$_2$PO$_4$}                           ...  
2  \mono{Agar}{20g}\\mono{ Distilled water}{980mL...  


In [4]:
df = clean_df(df)
print_verification(df)

=== CLEANING VERIFICATION ===

  sfi                          0
  hspace                       0
  Mix_tag                      0
  mu_tag                       0
  cdot_tag                     0
  double_curly                 61  <-- CHECK
  double_backslash             0
  corrupted_v                  0
  unclosed_amount              0
  unit_mL                      0
  unit_vg                      0
  html_entity                  0
  plain_reference              0
  mono_valid_inner_braces      79
  mono_unparseable             78  <-- CHECK

Unparseable \mono lines (78):
  '\\mono{{N}-Acetyl-D-glucosamine} {1.0}{g}'
  '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}'
  '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}'
  '\\mono{{myo}-Inositol} {5.0}{mg}'
  '\\mono{{p}-Aminobenzoic acid} {5.0}{mg}'
  '\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
  '\\mono{{p}-Aminobenzoic acid} {100.0}{mg}'
  '\\mono{{p}-Aminobenzoic acid} {50.0}{mg}'
  '\\mono{{n}-Butyric acid} {0.4}{ml}'
  '\\mono{

In [5]:
result = cleaning.verify(df)
# the counter is aggregate; to list them:
for tex in df["tex_text_clean"]:
    for line in tex.splitlines():
        if "\\mono" in line and not cleaning.MONO_FULL.search(line):
            print(repr(line.strip()[:90]))

'\\mono{{N}-Acetyl-D-glucosamine} {1.0}{g}'
'\\mono{{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{{myo}-Inositol} {5.0}{mg}'
'\\mono{{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{2.5% {N}-Acetyl-D-glucosamine solution} {20.0}{ml}'
'\\mono{{p}-Aminobenzoic acid} {100.0}{mg}'
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{{p}-Aminobenzoic acid} {50.0}{mg}'
'\\mono{Concentrated {Vibrio} suspension (see below)} {10.0}{ml}'
'\\mono{{n}-Butyric acid} {0.4}{ml}'
'\\mono{{iso}-Butyric acid} {0.4}{ml}'
'\\mono{{n}-Valeric acid} {0.2}{ml}'
'\\mono{{iso}-Valeric acid} {0.2}{ml}'
'\\mono{10% {N}-Acetyl-D-glucosamine solution*} {10.0}{ml}'
'\\mono{MnSO$_4$·{x}H$_2$O} {4.5}{mg}'
'\\mono{{p}-Aminobenzoic acid} {5.0}        {mg}'
'\\mono{MnSO$_4$·{x}H$_2$O} {1.0}{g}'
'\\mono{Cr$_2$(SO$_4$)$_3$·{x}H$_2$O} {0.5}{g}'
'\\mono{{p-}Aminobenzoic acid} {0.25}{mg}'
'\\mono{{p}-Aminobenzoic acid} {5.0}{mg}'
'\\mono{{p}-Aminobenzoic acid} {10.0}{mg}'
'\\mono{{n}--Valeric acid} {

In [6]:
def brace_balance(s):
    return s.count("{") - s.count("}")

before = df["tex_text"].apply(brace_balance)
after  = df["tex_text_clean"].apply(brace_balance)

changed = df[before != after]
print(f"Mediums whose brace balance changed: {len(changed)}")
print((after != 0).sum(), "mediums have unbalanced braces after cleaning")

Mediums whose brace balance changed: 33
28 mediums have unbalanced braces after cleaning


In [7]:
before = df["tex_text"].apply(brace_balance)
after  = df["tex_text_clean"].apply(brace_balance)

print("unbalanced BEFORE:", (before != 0).sum())
print("unbalanced AFTER :", (after != 0).sum())
print()
print("we FIXED   (bad→good):", ((before != 0) & (after == 0)).sum())
print("we BROKE   (good→bad):", ((before == 0) & (after != 0)).sum())
print("still bad  (bad→bad) :", ((before != 0) & (after != 0)).sum())

unbalanced BEFORE: 59
unbalanced AFTER : 28

we FIXED   (bad→good): 31
we BROKE   (good→bad): 0
still bad  (bad→bad) : 28


In [8]:
suspects = df[(before == 0) & (after != 0)]

for _, row in suspects.head(3).iterrows():
    trace = cleaning.clean_text_traced(row["tex_text"])
    prev = 0
    print(f"\n--- grmd {row['grmd']} ---")
    for name, text in trace.items():
        bal = brace_balance(text)
        mark = "  <-- HERE" if bal != prev else ""
        print(f"  {name:32} {bal:+d}{mark}")
        prev = bal

In [9]:
print("mono_unparseable:", cleaning.verify(df)["mono_unparseable"])

mono_unparseable: 78


In [10]:
import re
raw = df[df["grmd"] == 995]["tex_text"].values[0]
after_p1 = cleaning.pass1_formatting_tags(raw)

for line in after_p1.splitlines():
    if "{{" in line:
        print(repr(line.strip()[:100]))

'\\chu{{Solution 1:}}'
'\\chu{{Solution 2:}}'
'\\mono{{N}--Acetyl--D--glucosamine (Sigma)}               {2.0}{g}'
'\\chu{{Vitamin solution No. 6:}}'
'\\mono{{p}--Aminobenzoic acid}                           {10.0}{mg}'


In [11]:
import re
for grmd in [1332, 1207]:
    raw = df[df["grmd"] == grmd]["tex_text"].values[0]
    for line in raw.splitlines():
        if re.search(r"\\mono\{[^}]{1,10}\}-", line):
            print(f"{grmd} → {line.strip()[:90]!r}")

In [12]:
for grmd in [1332, 1207]:
    raw = df[df["grmd"] == grmd]["tex_text"].values[0]
    after = cleaning.pass1_formatting_tags(raw)
    after = cleaning.pass1_sfi(after)
    for line in after.splitlines():
        if re.search(r"\\mono\{[^}]{1,10}\}-", line):
            print(f"{grmd} → {line.strip()[:90]!r}")

1332 → '\\mono{2.5% {N}-Acetyl-D-glucosamine solution}      {20.0}{ml}'
1207 → '\\mono{10% {N}-Acetyl-D-glucosamine solution*}            {10.0}{ml}'


In [13]:
def split_mono(line):
    """Extract (name, amount, unit) from a \\mono line, respecting nesting.

    Returns None if the line cannot be read.
    """
    i = line.find("\\mono")
    if i == -1:
        return None
    j = line.find("{", i)
    if j == -1:
        return None

    depth = 0
    for k in range(j, len(line)):
        if line[k] == "{":
            depth += 1
        elif line[k] == "}":
            depth -= 1
            if depth == 0:
                name, rest = line[j + 1:k], line[k + 1:]
                break
    else:
        return None

    vals = re.findall(r"\{([^{}]*)\}", rest)
    if len(vals) < 2:
        return None
    return name.strip(), vals[0].strip(), vals[1].strip()

In [14]:
bad = []
for tex in df["tex_text_clean"]:
    for line in tex.splitlines():
        if "\\mono" in line and split_mono(line) is None:
            bad.append(line.strip())
print("unreadable:", len(bad))

unreadable: 0


In [15]:
unbalanced_ids = df.loc[
    df["tex_text_clean"].apply(brace_balance) != 0, "grmd"
].tolist()

In [16]:
def brace_balance(s):
    return s.count("{") - s.count("}")

bal = df["tex_text_clean"].apply(brace_balance)
unbalanced = df.loc[bal != 0, ["grmd", "md_name"]].copy()
unbalanced["balance"] = bal[bal != 0]

print(f"{len(unbalanced)} mediums with unbalanced braces\n")
print(unbalanced.to_string(index=False))

28 mediums with unbalanced braces

 grmd                                          md_name  balance
 1405            THIOHALORHABDUS METHYLOTROPHUS MEDIUM       -1
 1375                 MODIFIED LOW SALT MEDIUM FOR B11        1
 1347 FRESHWATER THIOSULFATE-OXIDIZING BACTERIA MEDIUM        1
 1331               HYDROGENOTROPHIC METHANOGEN MEDIUM       -1
 1326                    MODIFIED DESULFOVIBRIO MEDIUM       -2
 1227                              ZESTOSPHAERA MEDIUM        1
 1205         FUNDIDESULFOVIBRIO MAGNETOTACTCUS MEDIUM       -1
 1209            HALANAEROBIUM HYDROGENIFORMANS MEDIUM        1
 1179                                         M MEDIUM        1
 1175                          CAENICOLA MOBILE MEDIUM        1
 1167                                     MMFC1 MEDIUM        1
 1155             DESULFOOBULBUS OLIGOTROPHICUS MEDIUM        1
 1149                 METHANOMETHYLOPHILUS ALVI MEDIUM        1
 1074                        VULCANIBACILLUS B5 MEDIUM        1
 1014

In [133]:
import re

for _, row in df[df["grmd"].isin(unbalanced["grmd"])].iterrows():
    tags = set()
    for line in row["tex_text_clean"].splitlines():
        s = line.strip()
        if not s:
            continue
        if re.match(r"\\chu(SolutionB|SolutionC|JCM)", s):
            tags.add("bare_tag")
        elif brace_balance(line) != 0:
            tags.add("mono" if "\\mono" in s else "chu")
    print(f"{row['grmd']:>5}  {bal[row.name]:+d}  {sorted(tags)}")

 1405  -1  ['chu']
 1375  +1  ['chu']
 1347  +1  ['chu']
 1331  -1  ['chu']
 1326  -2  ['bare_tag']
 1227  +1  ['chu']
 1205  -1  ['bare_tag', 'chu']
 1209  +1  ['chu']
 1179  +1  ['chu']
 1175  +1  ['chu']
 1167  +1  ['chu']
 1155  +1  ['chu']
 1149  +1  ['chu']
 1074  +1  ['chu']
 1014  +1  ['chu']
  962  -1  ['chu']
  942  -1  ['chu']
  894  +1  ['chu']
  811  +1  ['chu']
  631  +1  ['chu']
  567  +1  ['chu']
  544  -1  ['chu']
  492  +2  ['chu']
  425  +1  ['chu']
  391  +1  ['chu']
  301  +1  ['chu']
  255  +1  ['chu']
   50  +1  ['chu']


In [17]:
def find_unclosed(tex):
    """Return (line_no, line) for each \\chu{ whose block never closes."""
    depth, opener = 0, None
    for n, line in enumerate(tex.splitlines()):
        for ch in line:
            if ch == "{":
                if depth == 0:
                    opener = (n, line.strip())
                depth += 1
            elif ch == "}":
                depth = max(0, depth - 1)
    return opener if depth > 0 else None

for grmd in [1353, 1398, 129, 50]:      # +3, +2, +2, +1
    tex = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    r = find_unclosed(tex)
    print(f"\n{grmd}: {r[1][:100]!r}" if r else f"\n{grmd}: closes cleanly (net from bare tags)")


1353: closes cleanly (net from bare tags)

1398: closes cleanly (net from bare tags)

129: closes cleanly (net from bare tags)

50: '\\chu{Cook or steam 20.0 g of oatmeal in 1.0 L of distilled'


In [18]:
for grmd in [1375, 1227, 1179, 492]:
    tex = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    r = find_unclosed(tex)
    print(f"{grmd}: {r[1][:90]!r}" if r else f"{grmd}: clean")

1375: '\\chu{Mix components thoroughly and autoclave under a N$_2$-CO$_2$ (4:1, v/v) gas mixture.'
1227: '\\chu{After the medium is cooled to room temperature, adjust pH to 6.0 -- 6.1.  Dispense th'
1179: '\\chu{After autoclaving, aseptically add filter-sterilized 2 M NaHCO$_3$ solution, 1 M Na$_'
492: '\\chu{Mix ingredients except sulfur, adjust pH to 7.0 with NaOH, and autoclave under a N$_2'


In [19]:
for grmd in [1375, 1227, 50]:
    tex = df[df["grmd"] == grmd]["tex_text_clean"].values[0]
    n, _ = find_unclosed(tex)
    print(f"\n=== {grmd} — from the unclosed opener ===")
    for line in tex.splitlines()[n:]:
        print(f"  {line.strip()!r}")


=== 1375 — from the unclosed opener ===
  '\\chu{Mix components thoroughly and autoclave under a N$_2$-CO$_2$ (4:1, v/v) gas mixture.'
  'Aseptically and anaerobically add the following solutions from anaerobic stocks'
  '(autoclaved or *filter-sterilized):'
  ''
  '\\mono{8% NaHCO$_3$ solution*} {25.0}{ml}'
  '\\mono{10% Glucose solution} {100.0}{ml}'
  '\\mono{Trace vitamins solution* (see Medium No. [284])} {2.0}{ml}'
  '\\mono{5% Na$_2$S·9H$_2$O solution} {6.0}{ml}'
  '\\chu{Adjust pH to 6.5-7.0, if necessary.}'
  ''

=== 1227 — from the unclosed opener ===
  '\\chu{After the medium is cooled to room temperature, adjust pH to 6.0 -- 6.1.  Dispense the medium into culture vessels (e.g., 10 ml in Balch type tubes) under the same gas mixture, seal with butyl rubber stoppers and autoclave.'
  ''

=== 50 — from the unclosed opener ===
  '\\chu{Cook or steam 20.0 g of oatmeal in 1.0 L of distilled'
  'water for 20 min. Filter through cheesecloth.'
  'Bring volume back up to 1.0 L and ad

In [20]:
print("BROKE:", ((before == 0) & (after != 0)).sum())
print("unreadable:", sum(
    1 for tex in df["tex_text_clean"]
      for line in tex.splitlines()
      if "\\mono" in line and split_mono(line) is None
))

BROKE: 1106
unreadable: 0


In [24]:
raw = pd.read_csv("/home/sheikh/Projects/Thesis/data/medium.csv")
clean = clean_df(raw)

before = raw["tex_text"].apply(brace_balance)
after  = clean["tex_text_clean"].apply(brace_balance)

print("unbalanced BEFORE:", (before != 0).sum())
print("unbalanced AFTER :", (after  != 0).sum())
print("we FIXED :", ((before != 0) & (after == 0)).sum())
print("we BROKE :", ((before == 0) & (after != 0)).sum())

unbalanced BEFORE: 59
unbalanced AFTER : 28
we FIXED : 31
we BROKE : 0


In [26]:
clean[["grmd", "md_name", "tex_text", "tex_text_clean"]].to_csv(
    "/home/sheikh/Projects/Thesis/data/medium_clean.csv", index=False
)